# Extract Codons Script Building

The goal is to make this more high throughput. 
This will work after creating the sam2pairwise output

### Input Requirement
Requires sam2pairwise output and SnpEFF Table

### Logic
1. We will create workable output from sam2pairwise. Output will include range and sequences. 
2. Going row by row, we will extract positions from SNPEff table, see if in range of sam2pair output, and grab only those details from that subset. 


In [79]:
# Load in the sam2pairwise output
s2pout = '/Users/kkwock/Documents/Github/VARify/data/test/ENCFF283TLK_chr12_66923_mapped_sam2pair.out'

# load in the snpeff table
snpeff = '/Users/kkwock/Documents/Github/VARify/data/test/ENCFF541HLI.snp_effects.chr12_only_filtergenic_snps.no_intron.no_utr.tsv'

In [80]:
import pandas as pd

s2p = pd.read_csv(s2pout, header=None) # need to ensure None header to work
snpeff_table = pd.read_csv(snpeff, sep='\t')

In [81]:
# Functions
import re

def sam2pair_extract(s2p_out):
    # Extract every 1st 2nd and 4th entry into separate DataFrames
    col1 = s2p_out.iloc[::4].reset_index(drop=True)
    col2 = s2p_out.iloc[1::4].reset_index(drop=True)
    col4 = s2p_out.iloc[3::4].reset_index(drop=True)

    # Split the data in column 1 by tab delimiter and keep elements at indices 1, 3, 4, and 9
    split_col1 = col1[0].str.split('\t', expand=True).iloc[:, [0, 1, 2, 3, 4, 5]]

    # Getting softclip number
    split_col1[5] = split_col1[5].apply(softclip_number)

    # Create a new DataFrame with the split data and col2_cleaned as columns
    result_df = pd.concat([split_col1, col2, col4], axis=1)

    # Rename the columns
    result_df.columns = ['QNAME', 'FLAG', 'CHR', 'POS', 'MAPQ', 'SOFTCLIP', 'SEQ', 'REF']

    result_df = result_df.dropna(subset=['QNAME'])

    # remove 'S' from REF ()
    result_df['REF'] = result_df.apply(lambda row: remove_softclip(row, 'REF'), axis=1)
    result_df['SEQ'] = result_df.apply(lambda row: remove_softclip(row, 'SEQ'), axis=1)

    # Get Range
    result_df['POS_END'] = result_df.apply(lambda row: int(row['POS']) + len(row['SEQ']), axis=1)

    # Interpret the SAM flag
    result_df['FLAG_EXPLAINED'] = result_df['FLAG'].astype(int).apply(sam_flag_explainer)
    
    #Drop unmapped
    result_df = result_df[result_df['FLAG_EXPLAINED'].apply(lambda x: x.get('read_unmapped', False) != True)]


    return result_df[['QNAME', 'FLAG', 'CHR', 'POS', 'POS_END', 'MAPQ', 'SOFTCLIP', 'SEQ', 'REF']]


def sam_flag_explainer(flag:int):
    """
    Intepret sam flag and make a flag-explained-dictionary. 

    """
    try:
        # Convert sam flags to binary format, like 0101001
        # flag is a integer.
        flag_binary = "{0:b}".format(flag)
    except Exception as err:
        if int(flag) == flag:
            flag = int(flag)
            flag_binary = "{0:b}".format(flag)
        else:
            print(f"{flag=}")
            raise ValueError from err

    flag_template = {
        "read_paired":False,
        "read_mapped_in_proper_pair":False,
        "read_unmapped":False,
        "mate_unmapped":False,
        "read_reverse_strand":False,
        "mate_reverse_strand":False,
        "first_in_pair":False,
        "second_in_pair":False,
        "not_primary_alignment":False,
        "read_fails_platform/vendor_quality_checks":False,
        "read_is_PCR_or_optical_duplicate":False,
        "supplementary_alignment":False,
    }

    for k, f in zip(flag_template.keys(), flag_binary[::-1]): # should intepret the flag from back.
        if int(f) == 1:
            flag_template[k] = True
    
    return flag_template


def softclip_number(input_string):
    pattern = r'^(\d+)S'

    # Use re.match to find the pattern at the beginning of the string
    match = re.match(pattern, input_string)

    if match:
        # Extract the matched part (the number before 'S')
        result = match.group(1)
        return result

    else: 
        return "0"
    
def remove_softclip(row, type = ['SEQ', 'REF']):
    s_value = int(row['SOFTCLIP'])
    ref_value = row[type]
    
    if s_value: 
        return ref_value[s_value:]
    else: 
        return ref_value

In [82]:
result_df = sam2pair_extract(s2p)

In [83]:
result_df.head()

,QNAME,FLAG,CHR,POS,POS_END,MAPQ,SOFTCLIP,SEQ,REF
0,8187f2d2-eced-496f-8977-2d09a7850fdc,0,chr12,43334,80306,33,9619,AAAACCTGTTTAGGGAACACACCAGCGTCTGT-CAGCTGG-TTTCA...,AAAACCTGTTTAGGGAACAC--CAGCATCTGTCCAGCTGGATTTCA...
1,03de73b9-126d-4fc6-a203-071c658b3463,16,chr12,43334,70243,33,3008,AAAACCTGTTTAGGGAACACCAGC-TGCTCATTCAGCTAGATTTTC...,AAAACCTGTTTAGGGAACACCAGCAT-CT-GTCCAGCTGGA-TTTC...
2,792eed51-d36d-424f-b4fd-48d8965f6bf4,0,chr12,43767,72609,32,1794,GTGATTCATGTACTGATCATGTTGTATAAGATCACTGGCTGGATGC...,GTGATTCATGTACTGATCATATTGTATAAGATCACTGGCTGGATGC...
3,de7ed9ca-222f-4bb7-999b-00638a46c4ba,16,chr12,43799,69822,17,13444,CACTGGCTGGATGCAGTGGCTCGTGCCTGTAAT-CC-------TGG...,CACTGGCTGGATGCAGTGGCTCGTGCCTGTAATCCCAACACTTTGG...
4,435913ed-4f68-45ce-bc6b-da71558a2c08,16,chr12,43923,69696,26,9499,AAAAATACAAAAATTAGCCAGGCATAGTGGTGCACGCCTGTAATCA...,AAAAATACAAAAATTAGCCAGGCATAGTGGTGCACGCCTGTAATCA...


In [84]:
snpeff_table.head()

,chr_id,snp_pos,ref_allele,alt_allele,gene_id,mrna_id,prot_id,strand,effect,snp_cds_pos,codon1_genome_pos,codon2_genome_pos,codon3_genome_pos,snp_aa_pos,ref_codon,alt_codon,ref_aa,alt_aa
0,chr12,66923,A,G,gene-IQSEC3,rna-NM_001170738.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C
1,chr12,66923,A,G,gene-IQSEC3,rna-XM_011520958.3,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C
2,chr12,66923,A,G,gene-IQSEC3,rna-XM_011520960.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C
3,chr12,66923,A,G,gene-IQSEC3,rna-XM_017019311.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C
4,chr12,66923,A,G,gene-IQSEC3,rna-XM_047428865.1,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C


In [85]:
# From snpeff_table, grab chr_id, snp_pos, codon1_genome_pos, codon2_genome_pos, codon3_genome_pos
sub = ['chr_id', 'snp_pos', 'codon1_genome_pos', 'codon2_genome_pos', 'codon3_genome_pos']
snpeff_df = snpeff_table[sub]
snpeff_df = snpeff_df.drop_duplicates(subset= sub)

In [86]:
# Creating data
import json

# Initialize a dictionary to hold the structured data
structured_data = {}

# Iterate over DataFrame rows and organize data into structured format
for index, row in snpeff_df.iterrows():
    chr_id = row["chr_id"]
    snp_pos = row["snp_pos"]
    codon1_genome_pos = row["codon1_genome_pos"]
    codon2_genome_pos = row["codon2_genome_pos"]
    codon3_genome_pos = row["codon3_genome_pos"]
    
    # Check if the chromosome ID already exists in the structured data
    if chr_id not in structured_data:
        structured_data[chr_id] = {}
    
    # Check if the SNP position already exists under the chromosome ID
    if snp_pos not in structured_data[chr_id]:
        structured_data[chr_id][snp_pos] = {
            "codon1_genome_pos": codon1_genome_pos,
            "codon2_genome_pos": codon2_genome_pos,
            "codon3_genome_pos": codon3_genome_pos
        }

# Convert the structured data to JSON
json_data = json.dumps(structured_data, indent=4)

snpeff_json = json.loads(json_data)

In [87]:
snpeff_json['chr12']['66923']['codon1_genome_pos']

66922

In [99]:
def get_codons(row, snpeff_json, codons={}):
    ''' 
    1. Go through each s2p record 
    2. Check if CHR in chr_id 
    3. Check if snp_pos between POS:POS_END
        a. Yes: calculate positions and grab SNP
        b. No: Skip
    '''
    start = int(row['POS']) - 1
    chr_id = row['CHR']
    
    # Check if chr_id exists in the dictionary
    if chr_id not in codons:
        codons[chr_id] = {}
    
    # Consolidate the dictionaries into one
    for target in snpeff_json[chr_id]:
        if int(target) in range(int(row['POS']), int(row['POS_END'])):
            pos = int(target) - start
        
            pos1 = snpeff_json[chr_id][target]["codon1_genome_pos"] - start
            pos2 = snpeff_json[chr_id][target]["codon2_genome_pos"] - start
            pos3 = snpeff_json[chr_id][target]["codon3_genome_pos"] - start
            
            alt1 = row['SEQ'][get_pos(row, pos1)]
            alt2 = row['SEQ'][get_pos(row, pos2)]
            alt3 = row['SEQ'][get_pos(row, pos3)]
            
            # Check if target exists in the dictionary
            if target not in codons[chr_id]:
                codons[chr_id][target] = {'codons': []}
            # Add the codon to the target
            codons[chr_id][target]['codons'].append(f"{alt1}{alt2}{alt3}")
    
    # Convert codons to set to remove duplicates and then back to list
    for chr_id, targets in codons.items():
        for target, codon_info in targets.items():
            
            codons[chr_id][target]['codons'] = list(set(codon_info['codons']))
    return codons
            
def get_pos(x, target):   
    index = 0
    letter_count = 0
    for char in x['REF']:
        if char != "-":
            letter_count += 1
        if letter_count == target:
            break
        index += 1
    
    return index


In [ ]:
    # for chr_id, targets in codons.items():
    #     for target, codon_info in targets.items():
    #         #codons[chr_id][target]['codons'] = list(set(codon_info['codons']))
        
    #         unique_codons = codon_info['codons']
    #         total_codons = len(unique_codons)
    #         codon_counts = {}
    #         for c in unique_codons:
    #             codon_counts[c] = codon_counts.get(c, 0) + 1
    #         for c2, count in codon_counts.items():
    #             codon_info[c2] = {'percentage': count / total_codons * 100}
    #         codon_info.pop('codons')

In [100]:
# Apply get_codons function row-wise and update the codons dictionary
for index, row in result_df.iterrows():
    codons = get_codons(row, snpeff_json, codons={})

codons

{'chr12': {'66923': {'codons': ['TGC']}, '66954': {'codons': ['AAG']}}}

In [92]:
codons

{'chr12': {'66923': {'codons': ['TCC',
    '---',
    'TAC',
    'T-C',
    '-AC',
    'TGC',
    'TA-',
    'CAC']},
  '66954': {'codons': ['---',
    'AAG',
    'ATG',
    'GAC',
    '-AG',
    'AA-',
    'AAA',
    '-AC',
    'AGC',
    '-GC',
    '--C',
    '-GG',
    'AAC',
    '-A-']}}}

In [73]:
def varify_codons(snpeff_table, codons):
    chr_id = snpeff_table['chr_id']
    snp_pos = str(snpeff_table['snp_pos'])

    snpeff_table['varify_codons'] = "NA"

    if chr_id in codons.keys():
        if snp_pos in codons[chr_id].keys():
            snpeff_table['varify_codons'] = codons[chr_id][snp_pos]['codons']

    return snpeff_table

snpeff_table = snpeff_table.apply(varify_codons, codons=codons, axis=1)



In [74]:
snpeff_table.head()

,chr_id,snp_pos,ref_allele,alt_allele,gene_id,mrna_id,prot_id,strand,effect,snp_cds_pos,codon1_genome_pos,codon2_genome_pos,codon3_genome_pos,snp_aa_pos,ref_codon,alt_codon,ref_aa,alt_aa,varify_codons
0,chr12,66923,A,G,gene-IQSEC3,rna-NM_001170738.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]"
1,chr12,66923,A,G,gene-IQSEC3,rna-XM_011520958.3,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]"
2,chr12,66923,A,G,gene-IQSEC3,rna-XM_011520960.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]"
3,chr12,66923,A,G,gene-IQSEC3,rna-XM_017019311.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]"
4,chr12,66923,A,G,gene-IQSEC3,rna-XM_047428865.1,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]"


In [75]:
# check if alt_codon in varify_alt_codons list 
def is_varify(x):
    x['alt verified?'] = x.alt_codon in x.varify_codons
    return x

snpeff_table.apply(is_varify, axis=1)

,chr_id,snp_pos,ref_allele,alt_allele,gene_id,mrna_id,prot_id,strand,effect,snp_cds_pos,codon1_genome_pos,codon2_genome_pos,codon3_genome_pos,snp_aa_pos,ref_codon,alt_codon,ref_aa,alt_aa,varify_codons,alt verified?
0,chr12,66923,A,G,gene-IQSEC3,rna-NM_001170738.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]",True
1,chr12,66923,A,G,gene-IQSEC3,rna-XM_011520958.3,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]",True
2,chr12,66923,A,G,gene-IQSEC3,rna-XM_011520960.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]",True
3,chr12,66923,A,G,gene-IQSEC3,rna-XM_017019311.2,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]",True
4,chr12,66923,A,G,gene-IQSEC3,rna-XM_047428865.1,protein_id,+,nonsynonymous,41,66922,66923,66924,14,TAC,TGC,Y,C,"[TCC, ---, TAC, T-C, -AC, TGC, TA-, CAC]",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7630,chr12,133219299,G,A,gene-ANHX,rna-XM_011534822.4,protein_id,-,nonsynonymous,908,133219300,133219299,133219298,303,GCC,GTC,A,V,NA,False
7631,chr12,133219299,G,A,gene-ANHX,rna-NM_001372060.1,protein_id,-,nonsynonymous,1349,133219300,133219299,133219298,450,GCC,GTC,A,V,NA,False
7632,chr12,133219299,G,A,gene-ANHX,rna-XM_006719743.5,protein_id,-,nonsynonymous,1376,133219300,133219299,133219298,459,GCC,GTC,A,V,NA,False
7633,chr12,133219299,G,A,gene-ANHX,rna-NM_001191054.1,protein_id,-,nonsynonymous,1037,133219300,133219299,133219298,346,GCC,GTC,A,V,NA,False


In [78]:
codons['chr12'].keys()

dict_keys(['66923', '66954'])